# nano lab notebook: the Word Hunt hero seat

**What this is.** One executable record of the ~11M-parameter `nano/` policy: how it is built (architecture), what it is
trained on and how (data, training), how it acts at match time (inference), what its play actually looks like (sample
trajectories), and the numbers behind the three claims on the slide (`docs/talk-nano-slide.md`): gate, league, live learning.

Every figure below is produced by this notebook from the code in `nano/` and the artifacts in `nano/results/` and
`nano/checkpoints/d6_s0.pt`. This is the record of **`d6_s0`**, the Mac-trained first hero (now the fallback seat); the
default seat is `d6_lambda.pt` (A100, 1M boards, length curriculum: gate 14.3x / 0.991 valid, `nano/results/gate_d6_lambda.md`),
same architecture and interface. Cells that say **live** re-run something small on this machine (CPU) so the notebook stays
honest about what the code does today; cells that say **recorded** read the JSON written by the real runs.

Re-render from the repo root:

```
.venv/bin/python -m jupyter nbconvert --execute --to html --output lab_notebook.html nano/lab_notebook.ipynb
```

| section | source of truth |
|---|---|
| 1. Architecture | `nano/model.py` |
| 2. Data and labels | `nano/data.py`, `nano/solver.py` |
| 3. Training | `nano/train.py`, `nano/results/train_d6_s0.json` (recorded) + a live smoke run |
| 4. Inference | `nano/seat.py`, `nano/policy.py` |
| 5. Sample trajectories | `nano/rollout.py` instrumented |
| 6. Gate and league | `nano/gate.py`, `nano/results/gate_d6_s0.json`, `docs/league.json` |
| 7. Live learning | `nano/learn.py`, `nano/results/live_learning_curve.md` |

In [ ]:
import json, math, os, re, sys, time
from pathlib import Path

# run from the repo root so `import nano.*` and the data/ paths resolve
root = Path.cwd()
while not (root / "nano" / "model.py").exists() and root != root.parent:
    root = root.parent
os.chdir(root)
sys.path.insert(0, str(root))
print("repo root:", root)

import numpy as np
import torch
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch

torch.set_num_threads(4)
torch.manual_seed(0)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3, "axes.spines.top": False, "axes.spines.right": False})

from nano.solver import Solver, NEIGHBORS, MAX_LEN, MIN_LEN, path_word, score_word, load_common_ranks
from nano.data import (annotate, word_weight, letter_distribution, random_board, generate, legal_moves, TYPES,
                       sample_path, rows_from_info, pack)
from nano.model import (NanoAgent, ModelConfig, encode, to_device, compute_loss, SEQ_LEN, N_TILES, VOCAB_SIZE,
                        TOK_CLS, TOK_LETTER, TOK_TILE, TOK_LEN, _SEGS)
from nano.policy import StudentPolicy, RandomSwiper, board_to_ids
from nano.seat import NanoSeat, bench
from nano.rollout import play, summarize
from nano.train import pre_encode, split, DeviceData, train as train_loop, evaluate
from nano.device import torch_device

CKPT = "nano/checkpoints/d6_s0.pt"
CPU = torch.device("cpu")

t0 = time.time()
solver = Solver()
ranks = load_common_ranks(valid=solver.word_set)
letter_p = letter_distribution(solver.words)
model, extra = NanoAgent.load(CKPT)
model.eval()
tr_info = extra.get("train", {})
print(f"solver: {len(solver.words)} enable1 words of length {MIN_LEN}-{MAX_LEN}; {len(ranks)} of them have a common-word rank")
print(f"checkpoint: {CKPT}  depth {model.cfg.depth}  d_model {model.cfg.d_model}  params {model.n_params()/1e6:.2f}M  "
      f"trained {tr_info.get('steps')} steps on {tr_info.get('device')} in {tr_info.get('wall_sec',0)/60:.0f} min")
print(f"setup {time.time()-t0:.1f}s")

## 1. Architecture

`nano/model.py` is a pre-norm transformer with a single dial, `depth`: width \(= 64 \cdot \text{depth}\), heads \(= \text{depth}\)
(head dim 64), MLP 4x. Depth 6 is what plays on stage (10.99M parameters).

The input is a fixed 26-token sequence. There is no dictionary anywhere in the forward pass: the network sees the 16 letters,
where they sit on the grid, and the tiles it has already traced.

| position | token | embeddings summed into it |
|---|---|---|
| 0 | `[CLS]` | token + segment `CLS` + position |
| 1..16 | tile \(i\) | letter id + segment `TILE` + grid position \(i\) |
| 17 | path length | `LEN+k` token + segment `LEN` + position |
| 18..25 | path step \(j\) | tile-index token + **that tile's letter** + segment `PATH` + step position (padded, masked out) |

Three heads read the `[CLS]` state:

- `target_logits (B,16)` — a pointer: \(\langle q(\text{cls}), k(h_i)\rangle / \sqrt{d}\) over the 16 tile states, with every tile that is
  not adjacent to the path head (or already used) masked to \(-\infty\). On an empty path it is "where to start".
- `type_logits (B,3)` — `{extend, submit, abort}`.
- `value_logit (B,)` — \(P(\text{this prefix still completes to a word})\); the "is this going somewhere" signal, and the
  escalation trigger for the stretch Gemma hand-off.

In [ ]:
# parameter budget by component, and the depth dial
def param_breakdown(m: NanoAgent) -> dict[str, int]:
    groups = {"embeddings (tok/seg/pos/letter)": 0, "attention blocks": 0, "MLP blocks": 0, "block layernorms": 0, "heads (type/pointer/value)": 0, "final layernorm": 0}
    for name, p in m.named_parameters():
        n = p.numel()
        if name.startswith(("tok.", "seg.", "pos.", "letter.")):
            groups["embeddings (tok/seg/pos/letter)"] += n
        elif ".attn." in name:
            groups["attention blocks"] += n
        elif ".mlp." in name:
            groups["MLP blocks"] += n
        elif ".ln1." in name or ".ln2." in name:
            groups["block layernorms"] += n
        elif name.startswith(("type_head", "q_target", "k_target", "value_head")):
            groups["heads (type/pointer/value)"] += n
        elif name.startswith("ln_f"):
            groups["final layernorm"] += n
    return groups

bd = param_breakdown(model)
depths = [2, 3, 4, 6, 8]
sweep = [(d, NanoAgent(ModelConfig(depth=d)).n_params()) for d in depths]

fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
names, vals = zip(*sorted(bd.items(), key=lambda kv: -kv[1]))
ax[0].barh(names, np.array(vals) / 1e6, color="#4C72B0")
for i, v in enumerate(vals):
    ax[0].text(v / 1e6 + 0.05, i, f"{v/1e6:.2f}M ({100*v/model.n_params():.0f}%)", va="center", fontsize=8)
ax[0].set_xlabel("parameters (M)"); ax[0].set_title(f"depth {model.cfg.depth}: {model.n_params()/1e6:.2f}M parameters by component"); ax[0].invert_yaxis()
ax[0].set_xlim(0, max(vals) / 1e6 * 1.45)
ax[1].plot([d for d, _ in sweep], [n / 1e6 for _, n in sweep], "o-", color="#DD8452")
for d, n in sweep:
    ax[1].annotate(f"{n/1e6:.1f}M", (d, n / 1e6), textcoords="offset points", xytext=(6, -2), fontsize=8)
ax[1].axvline(6, ls="--", color="gray", lw=1); ax[1].text(6.05, 1, "stage model", color="gray", fontsize=8)
ax[1].set_xlabel("depth (width = 64·depth, heads = depth)"); ax[1].set_ylabel("parameters (M)"); ax[1].set_title("the one dial")
plt.tight_layout(); plt.show()

print("vocab", VOCAB_SIZE, "tokens; seq len", SEQ_LEN, "; segments", int(_SEGS.max()) + 1)
print(model.blocks[0])

### 1.1 What one input looks like

Below: a board, a partial path, and the 26 tokens the model sees for it. The right panel is the legality mask `tmask`
that the pointer head is restricted to (green = legal next tile: adjacent to the path head and unused).

In [ ]:
BOARD = "ratehunslentgoid"      # the solver's own demo board (nano/solver.py __main__)
words_on = solver.words_on(BOARD)
print(f"board {BOARD!r}: {len(words_on)} words, max score {sum(score_word(w) for w in words_on)}; longest:",
      sorted(words_on, key=lambda w: (-len(w), w))[:8])

def draw_board(ax, board: str, path=(), title=None, tile_colors=None, cmap="Greens", vmin=None, vmax=None, annotate_vals=None, arrow_color="#C44E52"):
    board = board.lower()
    ax.set_xlim(-0.5, 3.5); ax.set_ylim(3.5, -0.5); ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    if tile_colors is not None:
        tc = np.asarray(tile_colors, dtype=float).reshape(4, 4)
        ax.imshow(tc, cmap=cmap, vmin=vmin if vmin is not None else np.nanmin(tc), vmax=vmax if vmax is not None else max(np.nanmax(tc), 1e-9), extent=(-0.5, 3.5, 3.5, -0.5))
    for i in range(16):
        r, c = divmod(i, 4)
        ax.add_patch(plt.Rectangle((c - 0.5, r - 0.5), 1, 1, fill=False, ec="#888", lw=1))
        ax.text(c, r - 0.12 if annotate_vals is not None else r, board[i].upper(), ha="center", va="center", fontsize=15, fontweight="bold", color="black")
        if annotate_vals is not None:
            ax.text(c, r + 0.3, annotate_vals[i], ha="center", va="center", fontsize=7, color="#333")
    for a, b in zip(path, path[1:]):
        (ra, ca), (rb, cb) = divmod(a, 4), divmod(b, 4)
        ax.add_patch(FancyArrowPatch((ca, ra), (cb, rb), arrowstyle="-|>", mutation_scale=14, lw=2.2, color=arrow_color, shrinkA=9, shrinkB=9))
    if path:
        r, c = divmod(path[0], 4)
        ax.add_patch(plt.Circle((c, r), 0.42, fill=False, ec=arrow_color, lw=2))
    if title:
        ax.set_title(title, fontsize=10)

# a partial path spelling "hun" (h=5, u=6, n=7 on this board? find it with the solver's own path finder)
from nano.data import word_paths
demo_path = word_paths(BOARD, "hun")[0]
enc = encode(board_to_ids(BOARD)[None], np.array([list(demo_path) + [-1] * (MAX_LEN - len(demo_path))], np.int8), np.array([len(demo_path)], np.uint8))

fig = plt.figure(figsize=(13, 3.8))
gs = fig.add_gridspec(1, 3, width_ratios=[1, 2.6, 1])
ax0 = fig.add_subplot(gs[0]); draw_board(ax0, BOARD, demo_path, title=f"path {demo_path} = '{path_word(BOARD, demo_path)}'")
ax1 = fig.add_subplot(gs[1]); ax1.grid(False)
seg_names = ["CLS", "TILE", "LEN", "PATH"]
seg_col = np.array(["#4C72B0", "#55A868", "#DD8452", "#C44E52"])
ids, segs, attn, letter = enc["ids"][0], enc["segs"][0], enc["attn"][0], enc["letter"][0]
for p in range(SEQ_LEN):
    tok = ids[p]
    if tok == TOK_CLS: lab = "[CLS]"
    elif TOK_LETTER <= tok < TOK_TILE: lab = chr(97 + tok - TOK_LETTER).upper()
    elif TOK_TILE <= tok < TOK_LEN: lab = f"t{tok - TOK_TILE}\n{chr(97 + letter[p]).upper()}"
    elif tok >= TOK_LEN: lab = f"len\n{tok - TOK_LEN}"
    else: lab = "pad"
    ax1.add_patch(plt.Rectangle((p, 0), 1, 1, color=seg_col[segs[p]], alpha=0.9 if attn[p] else 0.2))
    ax1.text(p + 0.5, 0.5, lab, ha="center", va="center", fontsize=7, color="white" if attn[p] else "#666")
    ax1.text(p + 0.5, -0.25, str(p), ha="center", va="center", fontsize=6, color="#666")
ax1.set_xlim(0, SEQ_LEN); ax1.set_ylim(-0.5, 1.4); ax1.set_yticks([]); ax1.set_xticks([])
for k, nm in enumerate(seg_names):
    ax1.add_patch(plt.Rectangle((k * 4 + 0.2, 1.1), 0.5, 0.25, color=seg_col[k])); ax1.text(k * 4 + 0.85, 1.22, nm, fontsize=8, va="center")
ax1.set_title("the 26 tokens (faded = padding, attention-masked)", fontsize=10)
ax2 = fig.add_subplot(gs[2]); draw_board(ax2, BOARD, demo_path, title="tmask: legal next tiles", tile_colors=enc["tmask"][0].astype(float), vmin=0, vmax=1.6)
plt.tight_layout(); plt.show()
print("legal next tiles:", [int(i) for i in np.flatnonzero(enc['tmask'][0])], "->", [BOARD[i].upper() for i in np.flatnonzero(enc['tmask'][0])])

### 1.2 What the heads say, and where `[CLS]` looks

Same input through the trained checkpoint: the three head outputs, then the last block's attention from `[CLS]` onto the
16 tile positions, one panel per head. The pointer is hard-masked to the legal tiles, so all of its mass sits on neighbours
of the path head.

In [ ]:
@torch.no_grad()
def forward_with_attention(m: NanoAgent, enc: dict) -> tuple[dict, list[torch.Tensor]]:
    b = to_device(enc, CPU)
    ids, segs, attn = b["ids"], b["segs"], b["attn"]
    B, L = ids.shape
    pos = torch.arange(L).unsqueeze(0)
    x = m.tok(ids) + m.seg(segs) + m.pos(pos)
    letter = b["letter"]
    x = x + m.letter(torch.where(letter >= 0, letter, torch.full_like(letter, 26)))
    kpm = ~attn
    weights = []
    for blk in m.blocks:
        h = blk.ln1(x)
        a, w = blk.attn(h, h, h, key_padding_mask=kpm, need_weights=True, average_attn_weights=False)
        weights.append(w)  # (B, heads, L, L)
        x = x + a
        x = x + blk.mlp(blk.ln2(x))
    x = m.ln_f(x)
    cls = x[:, 0]
    ht = x[:, 1:1 + N_TILES]
    tl = (m.k_target(ht) * m.q_target(cls).unsqueeze(1)).sum(-1) / math.sqrt(x.shape[-1])
    tl = tl.masked_fill(~b["tmask"], float("-inf"))
    return {"type_logits": m.type_head(cls), "target_logits": tl, "value_logit": m.value_head(cls).squeeze(-1)}, weights

out, W = forward_with_attention(model, enc)
type_p = torch.softmax(out["type_logits"], -1)[0].numpy()
tgt_p = np.nan_to_num(torch.softmax(out["target_logits"], -1)[0].numpy())
value = float(torch.sigmoid(out["value_logit"])[0])

fig, ax = plt.subplots(1, 3, figsize=(13, 3.6))
ax[0].bar(TYPES, type_p, color=["#55A868", "#4C72B0", "#C44E52"]); ax[0].set_ylim(0, 1); ax[0].set_title(f"type head   (value head: P(completes) = {value:.2f})")
for i, v in enumerate(type_p): ax[0].text(i, v + 0.02, f"{v:.2f}", ha="center")
draw_board(ax[1], BOARD, demo_path, title="pointer head: P(next tile)", tile_colors=tgt_p, cmap="Blues", vmin=0, vmax=max(tgt_p.max(), 1e-6) * 1.2, annotate_vals=[f"{p:.2f}" if p > 0.005 else "" for p in tgt_p])
# which words do those next tiles lead to?
top = np.argsort(-tgt_p)[:4]
lines = []
for t in top:
    if tgt_p[t] < 0.01: continue
    pre = path_word(BOARD, demo_path + (int(t),))
    conts = sorted([w for w in words_on if w.startswith(pre)], key=lambda w: ranks.get(w, 10**6))[:5]
    lines.append(f"{BOARD[t].upper()} ({tgt_p[t]:.2f}) -> {pre.upper()}...: {', '.join(conts) or '(nothing)'}")
ax[2].axis("off"); ax[2].text(0, 0.95, "what the top pointer choices lead to\n(solver, for the reader only; the model never sees it)\n\n" + "\n".join(lines), va="top", fontsize=9, family="monospace")
plt.tight_layout(); plt.show()

# attention from [CLS] onto tile positions, last block, per head
last = W[-1][0]  # (heads, L, L)
n_heads = last.shape[0]
fig, ax = plt.subplots(1, n_heads, figsize=(2.2 * n_heads, 2.5))
for h in range(n_heads):
    w_cls = last[h, 0].numpy()                     # attention row of [CLS]
    tiles = w_cls[1:1 + N_TILES]
    draw_board(ax[h], BOARD, demo_path, title=f"head {h}  (tiles {tiles.sum():.0%}, path {w_cls[18:].sum():.0%})", tile_colors=tiles, cmap="Purples", vmin=0, vmax=max(tiles.max(), 1e-6), arrow_color="#333")
fig.suptitle(f"block {len(W)-1} attention from [CLS] onto the 16 tiles (title: share of mass on tiles vs. path tokens)", fontsize=10, y=1.03)
plt.tight_layout(); plt.show()

## 2. Data and labels

Word Hunt gives what the AndroidWorld student never had: an exact oracle (the solver) and infinite boards. `nano/data.py`
turns that into **soft targets** rather than one-hot demonstrations, the "hindsight soft targets" rung that beat plain BC
in the prior work (`docs/nanoagent-handoff.md`).

The generative story behind a label: a player samples a word on the board with probability \(\propto\) `word_weight(word)`
(common words heavy; enable1-only words at weight 0.01), and traces one of its paths tile by tile. For every prefix along
the way the label is the *posterior of that process*:

- `target_dist[tile]` \(\propto\) mass of completions continuing through `tile`;
- `type_dist = [extend, submit, abort]` \(\propto\) `[mass of longer completions, weight of the prefix itself as a word, 0]`;
- `value = 1`.

Dead prefixes (a legal move no word continues) are mined beside the sampled ones with `type = abort`, `value = 0`, and an
all-zero pointer row that `soft_ce` skips.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.4))
r = np.arange(0, 30000)
ax[0].semilogy(r, 1.0 / (1.0 + r / 3000.0), label="common-30k word, by rank")
ax[0].axhline(0.01, color="#C44E52", ls="--", label="enable1-only word (0.01)")
ax[0].set_xlabel("rank in common-30k"); ax[0].set_ylabel("word_weight"); ax[0].set_title("frequency weighting of the words a 'player' picks"); ax[0].legend()
for w in ["the", "hunt", "lantern", "stoke", "aalii"]:
    if w in solver.word_set:
        ax[0].annotate(f"{w} ({word_weight(w, ranks):.3f})", (ranks.get(w, 29000), word_weight(w, ranks)), textcoords="offset points", xytext=(5, 5), fontsize=8)
ax[1].bar([chr(97 + i).upper() for i in range(26)], letter_p, color="#55A868")
ax[1].set_title("letter distribution boards are drawn from (letters of all enable1 3-8 words)"); ax[1].set_ylabel("P(letter)")
plt.tight_layout(); plt.show()
n_common = sum(1 for w in solver.words if w in ranks)
print(f"{n_common} of {len(solver.words)} playable words carry a common rank; the other {len(solver.words)-n_common} share weight 0.01 each")

### 2.1 The soft targets on one board

`annotate(board)` walks the trie over the board once and returns, for every live prefix, its weight as a word and the
completion mass through each next tile. Below: the start-tile distribution (empty prefix), then the targets for two
prefixes of increasing length. Note how `submit` mass appears once the prefix is itself a word.

In [ ]:
info, n_word_paths = annotate(BOARD, solver, ranks)
print(f"{len(words_on)} distinct words along {n_word_paths} word paths -> {len(info)} live prefixes on {BOARD!r}")

def label_for(prefix: tuple[int, ...]):
    w_word, kids = info[prefix]
    tgt = np.zeros(16); ext = 0.0
    for j, m in kids.items():
        tgt[j] = m; ext += m
    if ext > 0: tgt /= ext
    ty = np.array([ext, w_word, 0.0]); ty /= ty.sum()
    return tgt, ty

prefixes = [(), word_paths(BOARD, "hun")[0], word_paths(BOARD, "hunt")[0]]
fig, ax = plt.subplots(2, len(prefixes), figsize=(4 * len(prefixes), 6.2), gridspec_kw={"height_ratios": [3, 1.2]})
for k, p in enumerate(prefixes):
    tgt, ty = label_for(p)
    nm = f"'{path_word(BOARD, p)}'" if p else "(empty)"
    draw_board(ax[0, k], BOARD, p, title=f"prefix {nm}: target_dist (pointer label)", tile_colors=tgt, cmap="Oranges", vmin=0, vmax=max(tgt.max(), 1e-6) * 1.2, annotate_vals=[f"{v:.2f}" if v > 0.005 else "" for v in tgt])
    ax[1, k].bar(TYPES, ty, color=["#55A868", "#4C72B0", "#C44E52"]); ax[1, k].set_ylim(0, 1.05); ax[1, k].set_title("type_dist label", fontsize=9)
    for i, v in enumerate(ty): ax[1, k].text(i, v + 0.03, f"{v:.2f}", ha="center", fontsize=8)
plt.tight_layout(); plt.show()

# the mass behind each start tile, in words
starts = info[()][1]
tot = sum(starts.values())
print("start-tile mass (top 6):", ", ".join(f"{BOARD[t].upper()}@{t}={m/tot:.2f}" for t, m in sorted(starts.items(), key=lambda kv: -kv[1])[:6]))
rng_demo = np.random.default_rng(1)
print("6 sampled 'player' words:", [path_word(BOARD, sample_path(info, rng_demo)) for _ in range(6)])

### 2.2 A small dataset, live

`generate()` is what `python -m nano.data --boards 200000` runs with workers; here 300 boards on one worker so the shapes
and statistics are visible. The stage checkpoint trained on 200k boards → 4.96M samples (about 25 per board).

In [ ]:
t0 = time.time()
small = generate(300, n_paths=6, dead_frac=0.3, min_words=15, seed=123, workers=1)
n = len(small["value"]); per_board = np.bincount(small["bid"])
print(f"300 boards -> {n} samples ({n/300:.1f}/board) in {time.time()-t0:.1f}s; "
      f"dead fraction {1-small['value'].mean():.2f}; mean submit mass {small['ptype'][:,1].astype(float).mean():.3f}")
print({k: (v.shape, str(v.dtype)) for k, v in small.items()})

plen = small["plen"].astype(int); live = small["value"] == 1
fig, ax = plt.subplots(1, 3, figsize=(13, 3.3))
ax[0].hist(per_board, bins=20, color="#4C72B0"); ax[0].set_title("samples per board"); ax[0].set_xlabel("rows")
bins = np.arange(0, MAX_LEN + 2) - 0.5
ax[1].hist([plen[live], plen[~live]], bins=bins, stacked=True, color=["#55A868", "#C44E52"], label=["live prefix (value=1)", "mined dead prefix (value=0)"])
ax[1].set_title("prefix length of the samples"); ax[1].set_xlabel("path length"); ax[1].legend(fontsize=8)
sub = small["ptype"][:, 1].astype(float)
means = [sub[live & (plen == k)].mean() if (live & (plen == k)).any() else np.nan for k in range(MAX_LEN + 1)]
ax[2].plot(range(MAX_LEN + 1), means, "o-", color="#DD8452"); ax[2].set_ylim(0, 1); ax[2].set_title("mean submit mass in type_dist, by prefix length"); ax[2].set_xlabel("path length")
ax[2].axvline(MIN_LEN - 0.5, ls="--", color="gray", lw=1); ax[2].text(MIN_LEN - 0.4, 0.9, "3 letters = first legal submit", fontsize=8, color="gray")
plt.tight_layout(); plt.show()

## 3. Training

`nano/train.py` is the nanoagent recipe unchanged: soft cross-entropy on the type and pointer heads, BCE-with-logits on
value (weight 0.5), AdamW \((\beta_1, \beta_2) = (0.9, 0.95)\), weight decay 0.1, grad-norm clip 1.0, cosine to 10% of the
peak lr, **fp32 with no AMP** (bf16 autocast only ever on CUDA, kept off so records stay comparable). Peak lr follows
\(3\times10^{-4}\,\sqrt{4/\text{depth}}\) (smaller models take a larger lr). Non-finite losses are counted and skipped.

The whole dataset is tokenised once (`pre_encode`) and kept on the device as small-dtype tensors, so a step is pure indexing
plus the forward/backward. Validation holds out the last 2% of *boards* (samples of a board are contiguous), so no board leaks.

### 3.1 The recorded run behind `d6_s0.pt`

`runs/d6_s0` → `nano/results/train_d6_s0.json`: depth 6, batch 256, `--budget-min 30` on the laptop's MPS. The cosine
schedule is driven by the time budget, so the run finishes its anneal at exactly 30 minutes regardless of step count.

In [ ]:
T = json.load(open("nano/results/train_d6_s0.json"))
curve, vcurve = T["curve"], T["val_curve"]
steps = np.array([c["step"] for c in curve])
print(f"depth {T['depth']}  params {T['n_params']/1e6:.2f}M  device {T['device']}  batch {T['batch_size']}  peak lr {T['lr']:.2e}")
print(f"{T['steps']} steps in {T['wall_sec']/60:.1f} min = {T['samples_per_sec']:.0f} samples/s; {T['samples_seen']/1e6:.2f}M samples seen of {T['n_train']/1e6:.2f}M "
      f"(~{T['samples_seen']/T['n_train']:.2f} epochs); non-finite steps {T['nonfinite_steps']}")
print(f"loss {T['first_loss']:.3f} -> {T['final_loss']:.3f}; final val:", {k: round(v, 3) for k, v in T["val"].items() if k != "n"})

fig, ax = plt.subplots(2, 2, figsize=(13, 7))
ax[0, 0].plot(steps, [c["loss"] for c in curve], label="total (50-step mean)", color="black", lw=1.5)
for key, col in (("type", "#55A868"), ("target", "#4C72B0"), ("value", "#C44E52")):
    ax[0, 0].plot(steps, [c[key] for c in curve], label=key, color=col, alpha=0.7, lw=1)
ax[0, 0].plot([v["step"] for v in vcurve], [v["loss"] for v in vcurve], "o--", color="gray", ms=4, label="val total (50k rows)")
ax[0, 0].set_xscale("log"); ax[0, 0].set_xlabel("step"); ax[0, 0].set_title("training loss by head (soft-CE type, soft-CE pointer, 0.5·BCE value)"); ax[0, 0].legend(fontsize=8)
ax[0, 1].plot(steps, [c["lr"] for c in curve], color="#DD8452"); ax[0, 1].set_title("lr: cosine to 10%, driven by the 30-min budget"); ax[0, 1].set_xlabel("step")
ax2 = ax[0, 1].twinx(); ax2.plot(steps, [c["sec"] / 60 for c in curve], color="gray", ls=":", lw=1); ax2.set_ylabel("wall (min)", color="gray"); ax2.grid(False)
vs = [v["step"] for v in vcurve]
for key, col, lab in (("type_top1", "#55A868", "type top-1"), ("target_top1", "#4C72B0", "pointer top-1 (rows with a target)"), ("value_acc", "#C44E52", "value accuracy (>0)")):
    ax[1, 0].plot(vs, [v[key] for v in vcurve], "o-", color=col, ms=4, label=lab)
ax[1, 0].set_ylim(0.5, 1.0); ax[1, 0].set_xlabel("step"); ax[1, 0].set_title("validation accuracies (held-out boards)"); ax[1, 0].legend(fontsize=8)
ax[1, 1].plot(steps[1:], [c["loss"] for c in curve][1:], color="black", lw=1, label="train")
ax[1, 1].plot(vs, [v["loss"] for v in vcurve], "o--", color="gray", ms=4, label="val")
ax[1, 1].set_xlim(2000, T["steps"]); ax[1, 1].set_ylim(1.0, 1.35); ax[1, 1].set_xlabel("step"); ax[1, 1].set_title("zoom: last 15k steps, no train/val gap"); ax[1, 1].legend(fontsize=8)
plt.tight_layout(); plt.show()

**Reading it.** The pointer head carries most of the residual loss (0.82 of the 1.07 total): a soft target over several
valid continuations has irreducible entropy, so top-1 of 0.77 against the argmax of the label is the better read. The type head
is at 0.935 top-1 and the value head at 0.964 accuracy. Train and val sit on top of each other: with 4.4M samples seen on
4.96M available (under one epoch) there is nothing to overfit yet, so more steps or a second seed are the obvious next runs.

### 3.2 A live smoke run (CPU)

The same `train()` loop on the 300-board set from section 2, depth 4, 300 steps, batch 128, on CPU. This is what
`make train-smoke` does at smaller scale; it exists here so a reader can see the loop run end to end in under a minute.
The real run uses MPS (`make check-mps` first: `nano/device.py` refuses a silent CPU fallback on Apple Silicon).

In [ ]:
enc_small = pre_encode(small)
tr_np, va_np = split(enc_small, small["bid"], 0.1)
tr, va = DeviceData(tr_np, CPU), DeviceData(va_np, CPU)
print(f"train {tr.n} rows / val {va.n} rows")
logs = []
t0 = time.time()
smoke_model, smoke_info = train_loop(tr, va, depth=4, steps=300, batch_size=128, seed=0, log=logs.append, val_every=100)
print(f"{smoke_info['steps']} steps in {smoke_info['wall_sec']:.1f}s ({smoke_info['samples_per_sec']:.0f} samples/s on CPU, {torch.get_num_threads()} threads); "
      f"loss {smoke_info['first_loss']:.3f} -> {smoke_info['final_loss']:.3f}")
print("\n".join(logs[:1] + logs[-3:]))

fig, ax = plt.subplots(1, 2, figsize=(12, 3.4))
sc = smoke_info["curve"]
ax[0].plot([c["step"] for c in sc], [c["loss"] for c in sc], "o-", ms=3, label="live smoke: depth 4, 300 boards, batch 128, CPU")
head = [c for c in curve if c["step"] <= 300]
ax[0].plot([c["step"] for c in head], [c["loss"] for c in head], "s--", ms=3, color="gray", label="recorded d6_s0: first 300 steps, 200k boards, batch 256, MPS")
ax[0].set_xlabel("step"); ax[0].set_title("loss, first 300 steps"); ax[0].legend(fontsize=8)
vc = smoke_info["val_curve"]
for key, col in (("type_top1", "#55A868"), ("target_top1", "#4C72B0"), ("value_acc", "#C44E52")):
    ax[1].plot([v["step"] for v in vc], [v[key] for v in vc], "o-", color=col, label=key)
ax[1].set_ylim(0, 1); ax[1].set_title("smoke-run val accuracies (30 held-out boards)"); ax[1].set_xlabel("step"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

## 4. Inference

At match time `NanoSeat` (`nano/seat.py`) wraps `StudentPolicy` (`nano/policy.py`) on CPU with one thread, inside the game
server process. Per hand tick it gets `(board, path, found)` and returns `("extend", tile)`, `("submit",)` or `("abort",)`.
Decoding is independent per head: sample the type (temperature 1.0, after zeroing illegal types: no `submit` under 3 letters,
no `extend` with no legal tile), then sample a tile from the pointer restricted to legal tiles. Words already accepted by
the room get their `submit` mass zeroed so the seat does not re-trace them. Two seats with different seeds diverge.

The slide says **2 ms a move**; `bench()` measures it here.

In [ ]:
speed = {th: bench(CKPT, n=300, threads=th) for th in (1, 4)}
for th, s in speed.items():
    print(f"threads {th}: {s['ms_per_action']:.2f} ms/action (policy), {s['wall_ms_per_action']:.2f} ms wall, {s['params_M']}M params")

# per-action latency distribution, 1 thread, on a real board
torch.set_num_threads(1)
seat = NanoSeat(CKPT, temperature=1.0, seed=0, threads=1)
seat.reset(BOARD)
lat = []
for _ in range(400):
    t0 = time.perf_counter(); seat.step(); lat.append((time.perf_counter() - t0) * 1000)
lat = np.array(lat[20:])
torch.set_num_threads(4)

fig, ax = plt.subplots(1, 2, figsize=(12, 3.4))
ax[0].hist(lat, bins=40, color="#4C72B0"); ax[0].axvline(np.median(lat), color="#C44E52", ls="--", label=f"median {np.median(lat):.2f} ms")
ax[0].axvline(np.percentile(lat, 99), color="#DD8452", ls=":", label=f"p99 {np.percentile(lat, 99):.2f} ms"); ax[0].legend(fontsize=8)
ax[0].set_xlabel("ms per action (CPU, 1 thread)"); ax[0].set_title("latency of one decision")
seats = ["nano d6 (11M)\n1 thread", "hand tick\n(8-10 Hz)", "human reaction", "Gemma 12B text\n(mlx, per call)"]
ms = [speed[1]["ms_per_action"], 100, 300, 2100]
ax[1].bar(seats, ms, color=["#4C72B0", "gray", "#55A868", "#DD8452"]); ax[1].set_yscale("log"); ax[1].set_ylabel("ms (log)")
for i, v in enumerate(ms): ax[1].text(i, v * 1.15, f"{v:g} ms", ha="center", fontsize=8)
ax[1].set_title(f"time per decision: the nano is {2100/speed[1]['ms_per_action']:.0f}x under the 12B's call latency")
plt.tight_layout(); plt.show()

### 4.1 Where would it start? The empty-path pointer and value

On an empty path the pointer head is "which tile to start from". Sweeping every tile as a one-letter prefix gives the value
head's estimate of \(P(\text{completes})\) for each start; the solver's completion mass is shown beside it for the reader only.

In [ ]:
student = StudentPolicy(model, CPU, temperature=1.0, seed=0)
d0 = student.dists([BOARD], [()])[0]
one = student.dists([BOARD] * 16, [(i,) for i in range(16)])
val1 = np.array([d["value"] for d in one])
truth = np.zeros(16)
for t, m in info[()][1].items(): truth[t] = m
truth /= truth.sum()

fig, ax = plt.subplots(1, 3, figsize=(12.5, 3.8))
draw_board(ax[0], BOARD, title="model: P(start tile), empty path", tile_colors=d0["target"], cmap="Blues", vmin=0, vmax=d0["target"].max() * 1.2, annotate_vals=[f"{p:.2f}" if p > 0.005 else "" for p in d0["target"]])
draw_board(ax[1], BOARD, title="label: solver completion mass by start", tile_colors=truth, cmap="Oranges", vmin=0, vmax=truth.max() * 1.2, annotate_vals=[f"{p:.2f}" if p > 0.005 else "" for p in truth])
draw_board(ax[2], BOARD, title="model: value head after one tile", tile_colors=val1, cmap="Greens", vmin=0, vmax=1, annotate_vals=[f"{p:.2f}" for p in val1])
plt.tight_layout(); plt.show()
print(f"type dist on the empty path: {dict(zip(TYPES, np.round(d0['type'], 3)))}; value {d0['value']:.2f}")
print(f"pointer vs label: top-1 agree = {int(d0['target'].argmax()) == int(truth.argmax())}, L1 distance {np.abs(d0['target'] - truth).sum():.2f}")

## 5. Sample trajectories

`nano/rollout.py::play` runs a policy for a tick budget (600 actions = 75 s at 8 Hz) and scores it against the solver.
Below is the same loop instrumented to keep every episode: the tile path, the value head at each step, and how it ended
(`valid` submit, `invalid` submit of a non-word, `dup` re-submit of a word already found, or `abort`).

In [ ]:
def play_traced(board: str, policy, ticks: int = 600):
    policy.reset()
    words_on = solver.words_on(board)
    found: set[str] = set()
    episodes, cur = [], {"path": (), "values": [], "confs": []}
    for t in range(ticks):
        (atype, tile), inf = policy.act(board, cur["path"])
        cur["values"].append(float(inf.get("value", np.nan))); cur["confs"].append(float(inf.get("confidence", np.nan)))
        if atype == "extend" and tile >= 0:
            cur["path"] = cur["path"] + (tile,)
            continue
        w = path_word(board, cur["path"])
        if atype == "submit":
            end = "dup" if w in found else ("valid" if w in words_on else "invalid")
            if end == "valid": found.add(w)
        else:
            end = "abort"
        episodes.append({"tick": t, "word": w, "path": cur["path"], "end": end, "values": cur["values"], "confs": cur["confs"]})
        cur = {"path": (), "values": [], "confs": []}
    return {"board": board, "episodes": episodes, "found": found, "words_on": words_on, "score": sum(score_word(w) for w in found)}

student = StudentPolicy(model, CPU, temperature=1.0, seed=0)
tr1 = play_traced(BOARD, student, ticks=600)
eps = tr1["episodes"]
ends = {k: sum(e["end"] == k for e in eps) for k in ("valid", "invalid", "dup", "abort")}
print(f"board {BOARD!r}: {len(eps)} episodes in 600 actions; {ends}; found {len(tr1['found'])}/{len(tr1['words_on'])} words, score {tr1['score']} of {sum(score_word(w) for w in tr1['words_on'])}")
print("found:", sorted(tr1["found"], key=lambda w: (-len(w), w)))

print("\nfirst 24 episodes (tick, traced letters, outcome, value head along the path):")
for e in eps[:24]:
    vs = " ".join(f"{v:.2f}" for v in e["values"])
    print(f"  t{e['tick']:3d}  {e['word'].upper():<9} {e['end']:<8} value: {vs}")

### 5.1 The finger on the board

The twelve longest distinct words the seat found on this board, drawn as the tile paths it actually traced (circle = start).

In [ ]:
valid_eps = {}
for e in eps:
    if e["end"] == "valid": valid_eps.setdefault(e["word"], e)
show = sorted(valid_eps.values(), key=lambda e: (-len(e["word"]), e["tick"]))[:12]
fig, ax = plt.subplots(2, 6, figsize=(15, 5.4))
for a, e in zip(ax.ravel(), show):
    draw_board(a, BOARD, e["path"], title=f"'{e['word']}'  t{e['tick']}  +{score_word(e['word'])}")
for a in ax.ravel()[len(show):]: a.axis("off")
plt.suptitle(f"nano d6_s0 on {BOARD!r}: longest words found, as traced", y=1.0); plt.tight_layout(); plt.show()

### 5.2 Hesitation: the value head along an episode

PLAN.md section 3 asks the seat to "hesitate, back out". The value head is that signal: it should fall as a path stops
leading anywhere, and stay high when a word is forming. Left: value vs. step within the episode, coloured by how the
episode ended. Right: how every episode ended, and how many actions each outcome costs.

Note the `dup` bar: the bare `StudentPolicy` (what `rollout.py` and the gate measure) has no memory, so it re-traces words
it already has. In the match `NanoSeat` zeroes `submit` mass on accepted words (`on_result`), so the gate numbers are
conservative for the seat that actually plays; the cell below the plot shows the difference on this board.

In [ ]:
cols = {"valid": "#55A868", "invalid": "#C44E52", "dup": "#DD8452", "abort": "#8172B2"}
fig, ax = plt.subplots(1, 3, figsize=(14, 3.8))
for end in ("valid", "abort", "invalid"):
    sel = [e for e in eps if e["end"] == end][:25]
    for i, e in enumerate(sel):
        ax[0].plot(range(len(e["values"])), e["values"], color=cols[end], alpha=0.35, lw=1, label=end if i == 0 else None)
ax[0].set_xlabel("step within episode (last step is the submit/abort)"); ax[0].set_ylabel("value head  P(completes)"); ax[0].set_ylim(0, 1.02)
ax[0].set_title("value along the path, first 25 episodes per outcome"); ax[0].legend(fontsize=8)
# mean value at the final step by outcome
final = {end: [e["values"][-1] for e in eps if e["end"] == end] for end in cols}
ax[1].boxplot([final[k] for k in cols if final[k]], tick_labels=[f"{k}\n(n={len(final[k])})" for k in cols if final[k]], patch_artist=True, medianprops={"color": "black"})
for patch, k in zip(ax[1].patches, [k for k in cols if final[k]]): patch.set_facecolor(cols[k]); patch.set_alpha(0.6)
ax[1].set_ylabel("value at the deciding step"); ax[1].set_title("the seat aborts when value is low")
lens = {end: [len(e["values"]) for e in eps if e["end"] == end] for end in cols}
ax[2].bar(list(cols), [sum(lens[k]) for k in cols], color=[cols[k] for k in cols])
for i, k in enumerate(cols): ax[2].text(i, sum(lens[k]) + 5, f"{len(lens[k])} eps\n{sum(lens[k])} acts", ha="center", fontsize=8)
ax[2].set_ylabel("actions spent"); ax[2].set_title("where the 600 actions went")
plt.tight_layout(); plt.show()

# same board, same 600 actions, through NanoSeat (remembers accepted words, so no re-tracing)
seat = NanoSeat(CKPT, temperature=1.0, seed=0, threads=1)
seat.reset(BOARD)
seat_found: set[str] = set(); seat_submits = seat_valid = 0
for _ in range(600):
    p_before = seat.path
    a = seat.step()
    if a[0] == "submit":
        seat_submits += 1
        w = path_word(BOARD, p_before)
        ok = w in tr1["words_on"]
        seat.on_result(w, ok)
        if ok: seat_valid += 1; seat_found.add(w)
torch.set_num_threads(4)
print(f"bare StudentPolicy: {len(tr1['found'])} words, score {tr1['score']}, {ends['dup']} duplicate submits")
print(f"NanoSeat (dedup):   {len(seat_found)} words, score {sum(score_word(w) for w in seat_found)}, valid rate {seat_valid/max(seat_submits,1):.3f}, "
      f"longest {max(seat_found, key=len)!r}")

### 5.3 What it finds vs. what is there, and two seats diverging

Left: the word-length profile of what the seat found against everything the solver knows is on the board. This is the
honest caveat on the slide: volume of 3-4 letter words, few 5s, no 7s. Right: two seats with different sampling seeds on
the same boards find overlapping but different word sets (Jaccard overlap), which is why two nanos on one board look like two players.

In [ ]:
avail = np.bincount([len(w) for w in tr1["words_on"]], minlength=MAX_LEN + 1)[MIN_LEN:]
got = np.bincount([len(w) for w in tr1["found"]], minlength=MAX_LEN + 1)[MIN_LEN:]
L = np.arange(MIN_LEN, MAX_LEN + 1)

boards_div = []
rng = np.random.default_rng(777)
while len(boards_div) < 3:
    b = random_board(rng, letter_p)
    if len(solver.words_on(b)) >= 15: boards_div.append(b)
pairs = []
for b in boards_div:
    fa = play(b, StudentPolicy(model, CPU, 1.0, seed=1), solver, 400)["found"]
    fb = play(b, StudentPolicy(model, CPU, 1.0, seed=2), solver, 400)["found"]
    sa, sb = set(fa), set(fb)
    pairs.append((b, len(sa), len(sb), len(sa & sb), len(sa | sb)))

fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
ax[0].bar(L - 0.2, avail, 0.4, label="on the board (solver)", color="lightgray"); ax[0].bar(L + 0.2, got, 0.4, label="found by nano in 600 actions", color="#4C72B0")
for i, (a, g) in enumerate(zip(avail, got)):
    if a: ax[0].text(L[i] + 0.2, g + 0.5, f"{g}/{a}", ha="center", fontsize=8)
ax[0].set_xlabel("word length"); ax[0].set_ylabel("words"); ax[0].set_title(f"coverage by length on {BOARD!r}"); ax[0].legend(fontsize=8)
x = np.arange(len(pairs))
ax[1].bar(x - 0.25, [p[1] for p in pairs], 0.25, label="seed 1 found", color="#4C72B0"); ax[1].bar(x, [p[2] for p in pairs], 0.25, label="seed 2 found", color="#55A868"); ax[1].bar(x + 0.25, [p[3] for p in pairs], 0.25, label="both", color="#DD8452")
ax[1].set_xticks(x); ax[1].set_xticklabels([f"{p[0][:8]}\n{p[0][8:]}\nJaccard {p[3]/max(p[4],1):.2f}" for p in pairs], fontsize=8); ax[1].set_title("two seeds, same board, 400 actions each"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

## 6. Gate and league (recorded)

The gate was pre-registered in PLAN.md section 3 before training: on 50 unseen boards, score \(\ge 4\times\) a random swiper
**and** a higher valid-submit rate than the Gemma E4B seat (proxy today: the Gemma 4 12B text-grid seat, 0.20 valid on
\(n=20\), `gemma_seat/results/eval_n20_seed0.md`). `python -m nano.gate` wrote `nano/results/gate_d6_s0.{md,json}`.

In [ ]:
G = json.load(open("nano/results/gate_d6_s0.json"))
S, R = G["student"], G["random"]
print(f"verdict {G['verdict']}: score {S['score_mean']:.0f} vs random {R['score_mean']:.0f} = {G['ratio']:.2f}x (>= 4x); "
      f"valid rate {S['valid_rate']:.3f} vs Gemma proxy 0.20; {S['found_mean']:.1f} words/board, mean len {S['mean_len']:.2f}, longest {S['longest']!r}; "
      f"{G['speed']['ms_per_action']} ms/action")

# per-board comparison, replayed live on the first 12 gate boards (the JSON keeps only the summaries)
gate_boards = G["boards"][:12]
st = StudentPolicy(model, CPU, temperature=1.0, seed=G["args"]["seed"])
rd = RandomSwiper(seed=G["args"]["seed"])
t0 = time.time()
rows_s = [play(b, st, solver, 600) for b in gate_boards]
rows_r = [play(b, rd, solver, 600) for b in gate_boards]
print(f"replayed {len(gate_boards)} gate boards in {time.time()-t0:.0f}s: nano {np.mean([r['score'] for r in rows_s]):.0f} vs random {np.mean([r['score'] for r in rows_r]):.0f} per board")

fig, ax = plt.subplots(1, 3, figsize=(14.5, 3.8))
labels = ["nano d6_s0", "random swiper", "Gemma 12B text\n(proxy for E4B)"]
scores = [S["score_mean"], R["score_mean"], 1780]; valid = [S["valid_rate"], R["valid_rate"], 0.20]
c3 = ["#4C72B0", "gray", "#DD8452"]
ax[0].bar(labels, scores, color=c3); ax[0].axhline(4 * R["score_mean"], ls="--", color="#C44E52"); ax[0].text(1.5, 4 * R["score_mean"] * 1.08, f"gate: 4x random = {4*R['score_mean']:.0f}", color="#C44E52", fontsize=8, ha="center")
for i, v in enumerate(scores): ax[0].text(i, v + 120, f"{v:.0f}", ha="center")
ax[0].set_title("score per board (50 unseen boards)")
ax[1].bar(labels, valid, color=c3); ax[1].axhline(0.20, ls="--", color="#C44E52"); ax[1].set_ylim(0, 1.05); ax[1].set_title("valid-submit rate (gate: > 0.20)")
for i, v in enumerate(valid): ax[1].text(i, v + 0.02, f"{v:.3f}", ha="center")
xs = np.arange(len(gate_boards))
ax[2].bar(xs - 0.2, [r["score"] for r in rows_s], 0.4, color="#4C72B0", label="nano"); ax[2].bar(xs + 0.2, [r["score"] for r in rows_r], 0.4, color="gray", label="random")
ax[2].plot(xs, [r["max_score"] for r in rows_s], "k_", ms=12, label="max possible (solver)")
ax[2].set_yscale("log"); ax[2].set_xticks(xs); ax[2].set_xticklabels([b[:4] + "…" for b in gate_boards], fontsize=7, rotation=45); ax[2].set_title("per board, first 12 gate boards (live replay)"); ax[2].legend(fontsize=8)
plt.tight_layout(); plt.show()

hist = {3: 1139, 4: 676, 5: 83, 6: 4, 7: 0, 8: 0}  # word-length histogram from gate_d6_s0.md (all 50 boards)
print("word-length histogram over the 50 gate boards:", hist, f"-> {sum(hist.values())} valid words, {100*(hist[3]+hist[4])/sum(hist.values()):.0f}% are 3-4 letters")

In [ ]:
# the 20-board AI-only league, docs/league.json (Gemma rows are recorded one-call-per-board evals)
LG = json.load(open("docs/league.json"))
rows = LG["rows"]
clean = lambda s: re.sub(r"\*", "", s)
fig, ax = plt.subplots(1, 3, figsize=(14.5, 3.8))
names = [clean(r["seat"]).replace(" (", "\n(") for r in rows]
colors = ["gray", "#9ecae1", "#4C72B0", "#DD8452", "#f3b083"]
ax[0].barh(names, [r["score"] for r in rows], color=colors); ax[0].invert_yaxis(); ax[0].set_title("league: score per board (20 boards)")
for i, r in enumerate(rows): ax[0].text(r["score"] + 100, i, f"{r['score']:.0f}", va="center", fontsize=8)
ax[1].barh(names, [r["valid_rate"] for r in rows], color=colors); ax[1].invert_yaxis(); ax[1].set_xlim(0, 1.1); ax[1].set_title("valid-word rate"); ax[1].set_yticklabels([])
for i, r in enumerate(rows): ax[1].text(r["valid_rate"] + 0.02, i, f"{r['valid_rate']:.2f}", va="center", fontsize=8)
ax[2].barh(names, [r["mean_len"] for r in rows], color=colors); ax[2].invert_yaxis(); ax[2].set_xlim(3, 4); ax[2].set_title("mean word length (Gemma finds longer words when right)"); ax[2].set_yticklabels([])
for i, r in enumerate(rows): ax[2].text(r["mean_len"] + 0.01, i, f"{r['mean_len']:.2f}" + (f"  longest {r['longest']!r}" if r.get("longest") else ""), va="center", fontsize=8)
plt.tight_layout(); plt.show()
for r in rows: print(f"{clean(r['seat']):<45} score {r['score']:>6.0f}  valid {r['valid_rate']:.2f}  len {r['mean_len']:.2f}  wpm {r['wpm']:.1f}  {r['latency']:<28} {r['params']}")

## 7. Live learning between rounds

The thesis line (PLAN.md section 3): every validated word from any seat, human or Gemma, goes into a CPU update during the
20 s rematch countdown. `nano/learn.py::OnlineLearner.update` converts the round's words into the same soft targets as
`data.py` (restricted to the known words, no dead mining), blends them 0.5/0.5 with the model's own predictions (hinted
self-distillation, so 15 words cannot rewrite a 200k-board prior), runs 50 BC steps at lr \(2\times10^{-5}\) mixed 1:1 with
replay from the base distribution, replays a fixed 20-board held-out set under common random numbers, and **rolls back** if
the held-out score drops more than 3%.

### 7.1 The recorded 10-round curve

`python -m nano.learn --rounds 10` with the solver's top-15 common words (length \(\ge 4\)) standing in for Gemma as the teacher.

In [ ]:
txt = open("nano/results/live_learning_curve.md").read()
rows_ll = []
for line in txt.splitlines():
    m = re.match(r"\|\s*(\d+)\s*\|\s*(yes|rollback)\s*\|\s*(\d+)\s*\|\s*([\d.]+)\s*\|\s*([\d.]+)\s*\|\s*(\d+)\s*\|\s*(?:([\d.]+) -> ([\d.]+))?\s*\|\s*([\d.]+)", line)
    if m:
        r_, kept, ho, wpb, vr, taught, rb, ra, sec = m.groups()
        rows_ll.append({"round": int(r_), "kept": kept == "yes", "held_out": float(ho), "words": float(wpb), "valid": float(vr), "taught": int(taught),
                        "recall_before": float(rb) if rb else None, "recall_after": float(ra) if ra else None, "seconds": float(sec)})
print(f"parsed {len(rows_ll)} rows; kept {sum(r['kept'] for r in rows_ll[1:])}/{len(rows_ll)-1} updates; "
      f"held-out {rows_ll[0]['held_out']:.0f} -> {[r for r in rows_ll if r['kept']][-1]['held_out']:.0f}; "
      f"recall mean {np.mean([r['recall_before'] for r in rows_ll[1:]]):.2f} -> {np.mean([r['recall_after'] for r in rows_ll[1:]]):.2f}; "
      f"update wall mean {np.mean([r['seconds'] for r in rows_ll[1:]]):.1f}s")

fig, ax = plt.subplots(1, 3, figsize=(14.5, 3.6))
rr = [r["round"] for r in rows_ll]
ax[0].plot(rr, [r["held_out"] for r in rows_ll], "o-", color="#4C72B0", label="held-out score (20 boards)")
for r in rows_ll[1:]:
    if not r["kept"]: ax[0].plot(r["round"], r["held_out"], "rx", ms=11, mew=2, label="rolled back" if r["round"] == [q["round"] for q in rows_ll if not q["kept"]][0] else None)
ax[0].fill_between(rr, rows_ll[0]["held_out"] * 0.97, rows_ll[0]["held_out"] * 1.03, color="gray", alpha=0.12, label="±3% tolerance band (from round 0)")
ax[0].set_xlabel("round"); ax[0].set_title("held-out stays flat; two rounds rolled back"); ax[0].legend(fontsize=8)
ax[1].plot(rr[1:], [r["recall_before"] for r in rows_ll[1:]], "o-", color="gray", label="before update")
ax[1].plot(rr[1:], [r["recall_after"] for r in rows_ll[1:]], "o-", color="#55A868", label="after update")
ax[1].set_ylim(0, 0.8); ax[1].set_xlabel("round"); ax[1].set_title("recall of the 15 taught words on the round board"); ax[1].legend(fontsize=8)
ax[2].bar(rr[1:], [r["seconds"] for r in rows_ll[1:]], color=["#55A868" if r["kept"] else "#C44E52" for r in rows_ll[1:]])
ax[2].axhline(20, ls="--", color="#C44E52"); ax[2].text(1, 20.6, "20 s rematch countdown", color="#C44E52", fontsize=8)
ax[2].set_ylim(0, 24); ax[2].set_xlabel("round"); ax[2].set_ylabel("s"); ax[2].set_title("update wall time on CPU (green kept, red rolled back)")
plt.tight_layout(); plt.show()

### 7.2 Three rounds, live

`simulate()` run here for three rounds against the same solver-as-teacher stand-in. There is no `data/wh_200000.npz` on this
machine, so the learner generates its 400 replay boards at construction (that is most of the setup time). Numbers move with
the seed; the point is the mechanism: update → held-out check → keep or roll back, inside the budget.

In [ ]:
from nano.learn import simulate
t0 = time.time()
live_rows = simulate(CKPT, rounds=3, out=None, seed=41_000)
print(f"\n{time.time()-t0:.0f}s total including learner construction")
for r in live_rows[1:]:
    print(f"round {r['round']}: {'KEEP' if r['kept'] else 'ROLLBACK'}  held-out {r['held_out_before']:.0f} -> {r['held_out_after']:.0f}  "
          f"taught {r['taught']} words {r['words'][:6]}...  recall {r['recall_before']:.2f} -> {r['recall_after']:.2f}  {r['seconds']:.1f}s")

## 8. Summary and open items

**The three stage numbers, as measured in this notebook and its recorded artifacts**

1. *11M parameters, zero tokens, ~2 ms a move.* Depth-6 transformer over 26 fixed tokens; pointer/type/value heads; no
   dictionary at inference. 17,278 steps × 256 on 200k self-solved boards in 30 min on MPS; `bench()` above on this CPU.
2. *Gate passed at 9.6x, 97% valid.* Pre-registered before training; 50 unseen boards; 9126 vs 952 per board against the
   random swiper, 0.971 valid-submit rate against the Gemma proxy's 0.20. League on 20 boards: nano 11245, 12B text 1780.
3. *It learns the words it was just shown.* 8/10 updates kept in ~9.5 s each; taught-word recall 0.35 → 0.46 while held-out
   moved −0.2%. It does not get better at Word Hunt in ten rounds; say exactly that.

**Caveats the plots make visible**

- Word length: 3.44 mean, ~96% of found words are 3-4 letters, nothing above 6 in 50 boards. Gemma finds the long words when it is right.
- The pointer head carries most of the loss; the soft target has irreducible entropy so its top-1 (0.77) is the better read.
- Train and val overlap with under one epoch seen: more steps and a second seed (`d6_s1`) are unspent budget, not tuning.
- Live-learning held-out noise is about ±4% at 300 actions × 20 boards, so the ±3% rollback band is close to the noise floor.

**Next runs, in order:** second seed for the gate spread; longer budget (60 min) on the same data; a length-aware sampling
temperature or a bonus on `submit` for ≥5-letter prefixes if the long-word gap matters on stage; human boards for the stretch
target (≥ 60% of human median) once the room has rounds.